In [1]:
import torch
import torchvision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

In [2]:
path = kagglehub.dataset_download("waalbannyantudre/hate-speech-detection-curated-dataset")

100%|██████████| 114M/114M [00:06<00:00, 18.8MB/s] 

Extracting files...


In [ ]:
print(path)

/root/.cache/kagglehub/datasets/waalbannyantudre/hate-speech-detection-curated-dataset/versions/1


In [4]:
import os
hate_speech_data = pd.read_csv(os.path.join(path, "HateSpeechDatasetBalanced.csv"))
print(hate_speech_data.head())
print(hate_speech_data['Label'].value_counts())

                                             Content  Label
0  denial of normal the con be asked to comment o...      1
1  just by being able to tweet this insufferable ...      1
2  that is retarded you too cute to be single tha...      1
3  thought of a real badass mongol style declarat...      1
4                                afro american basho      1
Label
1    364525
0    361594
Name: count, dtype: int64


In [5]:
!pip install transformers contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.7 MB/s eta 0:00:00


In [6]:
import nltk, spacy, re, string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer, WordNetLemmatizer
import contractions

nltk.download(['punkt_tab', 'wordnet', 'punkt', 'stopwords'], quiet=True)

stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

punctuation_pattern = re.compile(f"[{re.escape(string.punctuation)}]")
digit_pattern = re.compile(r'\d+')
whitespace_pattern = re.compile(r'\s+')
non_word_pattern = re.compile(r'\W+')

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    text = text.lower()
    text = contractions.fix(text)
    text = digit_pattern.sub(' ', text)
    text = punctuation_pattern.sub(' ', text)
    text = non_word_pattern.sub(' ', text)
    text = whitespace_pattern.sub(' ', text).strip()
    tokens = word_tokenize(text)
    res = []
    for token in tokens:
        if token not in stop_words and len(token) < 50:
            processed_token = stemmer.stem(token)
            res.append(processed_token)
    res_full = " ".join(res)
    return res_full
    
hate_speech_data['Cleaned_Content'] = hate_speech_data['Content'].apply(clean_text)

In [7]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(hate_speech_data, test_size=0.2, random_state=42, shuffle=True, stratify=hate_speech_data['Label'])

In [8]:
len(train_data), len(test_data)

(580895, 145224)

In [9]:
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import DistilBertModel, DistilBertTokenizer

from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding


class HateSpeechDataset(Dataset):
    
    def __init__(self, data):
        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')  
        enc = self.tokenizer(
            text=data["Cleaned_Content"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=64
        )

        self.input_ids = enc["input_ids"]
        self.attn = enc["attention_mask"]
        self.labels = data["Label"].tolist()      
        self.data = data
       
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attn[idx]),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }
    

In [10]:


tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')      



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Define Classifier model

In [ ]:
class HateSpeechClassifier(nn.Module):
    def __init__(self):
        super(HateSpeechClassifier, self).__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.avgpool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        # outputs = self.avgpool(outputs.transpose(1, 2)).squeeze(-1)
        cls_embedding = outputs[:, 0, :] 
        output = self.classifier(self.dropout(cls_embedding))
        return output

In [ ]:
train_dataset = HateSpeechDataset(train_data)
test_dataset = HateSpeechDataset(test_data)

from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch_size = 64
dataloader_train = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collator, shuffle=True, num_workers=2, pin_memory=True)
dataloader_test = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collator, shuffle=False, num_workers=2, pin_memory=True)   
model = HateSpeechClassifier()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
from tqdm import tqdm
train_losses = []
test_losses = []
num_epochs = 3

for epoch in range(num_epochs):
    print(f"epoch {epoch+1}/{num_epochs}")
    model.train()
    total_train_loss = 0
    for i, batch in enumerate(tqdm(dataloader_train)):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.squeeze(), labels.float())
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(dataloader_train)
    train_losses.append(avg_train_loss)

    model.eval()
    total_test_loss = 0
    preds = list()
    truths = list()
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader_test)):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.squeeze(), labels.float())
            total_test_loss += loss.item()
    avg_test_loss = total_test_loss / len(dataloader_test)
    test_losses.append(avg_test_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss}, Test Loss: {avg_test_loss}")


epoch 1/3


100%|██████████| 1135/1135 [03:46<00:00,  5.00it/s]


Epoch 1/3, Train Loss: 0.6848690448208615, Test Loss: 0.9087013862206548
epoch 2/3


 26%|██▌       | 1163/4539 [11:48<34:27,  1.63it/s]

## Traditional Transformer method

In [ ]:
!pip install torchtext

In [ ]:
import torchtext
from collections import Counter
# YOUR CODE HERE
tokenizer = torchtext.data.utils.get_tokenizer("basic_english")

def yield_tokens(df, text_column="Content"):
    for text in df[text_column]:
        yield tokenizer(text)

vocab = torchtext.vocab.build_vocab_from_iterator(yield_tokens(hate_speech_data), specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])

In [ ]:
def text_pipeline(x): 
    return vocab(tokenizer(x))

def get_text_lengths(data_iter):
    lengths = []
    for _, text in tqdm(data_iter):
        tokenized_text = tokenizer(text)
        lengths.append(len(tokenized_text))
    return np.array(lengths)

def collate_batch(batch, max_seq_len=256):
    labels, texts = zip(*batch)
    # the labels start at 1 but predictions start at 0. To align them, we modify lables
    labels = torch.tensor(labels,dtype=torch.long)-1
    
    text_list = []
    for text in texts:
        # Truncate or pad to max_seq_len
        tokenized_text = text_pipeline(text)
        if len(tokenized_text) > max_seq_len:
            tokenized_text = tokenized_text[:max_seq_len]  # Truncate if longer than max_seq_len
        else:
            # Pad if shorter than max_seq_len
            tokenized_text = tokenized_text + [vocab["<pad>"]] * (max_seq_len - len(tokenized_text))
        
        text_list.append(torch.tensor(tokenized_text, dtype=torch.long))
    
    padded_texts = torch.stack(text_list)  # Stack the sequences into a tensor
    return padded_texts, labels

In [ ]:
class SimpleHateSpeechDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data.iloc[idx]['Label'], self.data.iloc[idx]['Content']

In [ ]:
train_dataset = SimpleHateSpeechDataset(train_data)
test_dataset = SimpleHateSpeechDataset(test_data)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

label_counter = Counter()
for texts, labels in train_loader:
    label_counter.update(labels.tolist())
print("Label counts:", label_counter)

In [ ]:
class TransformerModel(torch.nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_encoder_layers, num_classes, dropout=0.1):
        
        super(TransformerModel, self).__init__()
        
        self.embedding = torch.nn.Embedding(vocab_size, embed_size)
        self.transformer = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(embed_size, num_heads, embed_size * 2, dropout),
            num_encoder_layers
        )
        self.fc = torch.nn.Linear(embed_size, num_classes)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x)  # Embedding layer
        x = x.permute(1, 0, 2)  # Transformer expects (seq_len, batch_size, embedding_size)
        x = self.transformer(x)  # Apply transformer
        x = x.mean(dim=0)  # Pooling (take the mean of all tokens in the sequence)
        x = self.dropout(x)
        x = self.fc(x)  # Final classification layer
        return x

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Initialize the model, 
embed_size = 16
num_heads = 2
num_encoder_layers = 2
num_classes = 4
model = TransformerModel(len(vocab), embed_size, num_heads, num_encoder_layers, num_classes)

# Initialize loss function, and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.to(device)

In [ ]:
num_epochs = 3
train_losses = np.zeros(num_epochs)
train_accuracies = np.zeros(num_epochs)

test_losses = np.zeros(num_epochs)
test_accuracies = np.zeros(num_epochs)

In [ ]:
# YOUR CODE HERE
def train_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    epoch_loss = 0
    epoch_accuracy = 0
    # total_batches = 0
    total_batches = len(train_loader)
    
    for texts, labels in tqdm(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(texts)
        
        # Compute loss and gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        # Update model parameters
        optimizer.step()
        
        # Calculate accuracy
        preds = torch.argmax(outputs, dim=1)
        correct = (preds == labels).sum().item()
        accuracy = correct / labels.size(0)
        
        epoch_loss += loss.item()
        epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches

def evaluate(model, test_loader, loss_fn):
    model.eval()
    epoch_loss = 0
    epoch_accuracy = 0
    total_batches = len(test_loader)
    correct_preds = []
    incorrect_preds = []

    with torch.no_grad():
        for texts, labels in tqdm(test_loader):
            texts, labels = texts.to(device), labels.to(device)
            # Forward pass
            outputs = model(texts)
            
            # Compute loss
            loss = loss_fn(outputs, labels)
            
            # Calculate accuracy
            preds = torch.argmax(outputs, dim=1)
            correct_mask = preds == labels
            correct = (preds == labels).sum().item()
            accuracy = correct / labels.size(0)
            for i in range(len(labels)):
                if correct_mask[i]:
                    correct_preds.append((texts[i].cpu(), labels[i].cpu(), preds[i].cpu()))
                else:
                    incorrect_preds.append((texts[i].cpu(), labels[i].cpu(), preds[i].cpu()))
            epoch_loss += loss.item()
            epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches, correct_preds, incorrect_preds

In [ ]:
# YOUR CODE HERE
def train_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    epoch_loss = 0
    epoch_accuracy = 0
    # total_batches = 0
    total_batches = len(train_loader)
    
    for texts, labels in tqdm(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(texts)
        
        # Compute loss and gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        # Update model parameters
        optimizer.step()
        
        # Calculate accuracy
        preds = torch.argmax(outputs, dim=1)
        correct = (preds == labels).sum().item()
        accuracy = correct / labels.size(0)
        
        epoch_loss += loss.item()
        epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches

def evaluate(model, test_loader, loss_fn):
    model.eval()
    epoch_loss = 0
    epoch_accuracy = 0
    total_batches = len(test_loader)
    correct_preds = []
    incorrect_preds = []

    with torch.no_grad():
        for texts, labels in tqdm(test_loader):
            texts, labels = texts.to(device), labels.to(device)
            # Forward pass
            outputs = model(texts)
            
            # Compute loss
            loss = loss_fn(outputs, labels)
            
            # Calculate accuracy
            preds = torch.argmax(outputs, dim=1)
            correct_mask = preds == labels
            correct = (preds == labels).sum().item()
            accuracy = correct / labels.size(0)
            for i in range(len(labels)):
                if correct_mask[i]:
                    correct_preds.append((texts[i].cpu(), labels[i].cpu(), preds[i].cpu()))
                else:
                    incorrect_preds.append((texts[i].cpu(), labels[i].cpu(), preds[i].cpu()))
            epoch_loss += loss.item()
            epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches, correct_preds, incorrect_preds

In [ ]:
# YOUR CODE HERE
import matplotlib.pyplot as plt

num_epochs = 3
epochs = list(range(0, num_epochs))

# Create 2x2 grid of subplots using plt.subplot
plt.figure(figsize=(10, 8))

# Training Loss
plt.subplot(2, 2, 1)  # (rows, columns, index)
plt.plot(epochs, train_losses, color='blue', label='Train Loss')
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Testing Loss
plt.subplot(2, 2, 2)
plt.plot(epochs, test_losses, color='orange', label='Test Loss')
plt.title('Testing Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Training Accuracy
plt.subplot(2, 2, 3)
plt.plot(epochs, train_accuracies, color='green', label='Train Accuracy')
plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Testing Accuracy
plt.subplot(2, 2, 4)
plt.plot(epochs, test_accuracies, color='red', label='Test Accuracy')
plt.title('Testing Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Adjust layout and display
plt.tight_layout()
plt.show()